# Notebook 3: Valuation

**Project:** Automated Equity Research Platform

This notebook runs independently from the other notebooks.

## Standalone setup

This notebook is designed to run by itself. It can use a local clone, a Google Drive folder, or a GitHub repository.

Before publishing, replace `YOUR_USERNAME` in `REPO_URL` with your actual GitHub username.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

PROJECT_NAME = "Automated_Equity_Research_Platform"
REPO_URL = "https://github.com/YOUR_USERNAME/Automated_Equity_Research_Platform.git"

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception:
        pass

candidate_roots = [
    Path.cwd(),
    Path("/content") / PROJECT_NAME,
    Path("/content/drive/MyDrive") / PROJECT_NAME,
    Path("/content/drive/MyDrive/AI_Equity_Research_Platform"),
]

PROJECT_ROOT = next(
    (path for path in candidate_roots if (path / "pyproject.toml").exists()),
    None,
)

if PROJECT_ROOT is None:
    if "YOUR_USERNAME" in REPO_URL:
        raise RuntimeError(
            "Project files were not found. Upload the full project folder to Google Drive "
            "or replace YOUR_USERNAME in REPO_URL with your real GitHub username."
        )
    target = Path("/content") / PROJECT_NAME
    if not target.exists():
        subprocess.run(["git", "clone", REPO_URL, str(target)], check=True)
    PROJECT_ROOT = target

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

print("Project root:", PROJECT_ROOT)

In [ ]:
%pip install -q -r requirements.txt
%pip install -q -e .

In [ ]:
import os

try:
    from google.colab import userdata
    for secret_name in ("SEC_USER_AGENT", "FRED_API_KEY"):
        try:
            value = userdata.get(secret_name)
            if value:
                os.environ[secret_name] = value
        except Exception:
            pass
except Exception:
    pass

print("SEC enabled:", bool(os.getenv("SEC_USER_AGENT")))
print("FRED enabled:", bool(os.getenv("FRED_API_KEY")))

Run DCF valuation, sensitivity analysis, and peer comparable valuation.

In [ ]:
from equity_research.data import YahooFinanceClient
from equity_research.fundamentals import FundamentalAnalyzer
from equity_research.valuation import DCFInputs, DCFModel, ComparableValuation
from equity_research.visualization import dcf_projection_chart, sensitivity_heatmap
import pandas as pd

TICKER = input("Ticker: ").strip().upper() or "AAPL"
peer_input = input("Peers separated by commas: ").strip()
PEERS = [p.strip().upper() for p in peer_input.split(",") if p.strip()] or ["MSFT", "GOOGL", "AMZN", "META"]

fcf_growth = float(input("FCF growth decimal, example 0.08: ") or 0.08)
discount_rate = float(input("Discount rate decimal, example 0.09: ") or 0.09)
terminal_growth = float(input("Terminal growth decimal, example 0.025: ") or 0.025)

yahoo = YahooFinanceClient()
bundle = yahoo.bundle(TICKER, start="2016-01-01")
snapshot = FundamentalAnalyzer().analyze(bundle)
assumptions = DCFInputs(initial_growth=fcf_growth, discount_rate=discount_rate, terminal_growth=terminal_growth)

dcf_model = DCFModel()
try:
    dcf = dcf_model.value(snapshot, assumptions)
    display(dcf.projection_frame())
    display(pd.Series(dcf.to_dict()).to_frame("Value"))
    dcf_projection_chart(dcf).show()
    sensitivity = dcf_model.sensitivity(snapshot, base_inputs=assumptions)
    display(sensitivity)
    sensitivity_heatmap(sensitivity, TICKER).show()
except Exception as exc:
    dcf = None
    print("DCF skipped:", type(exc).__name__, exc)

In [ ]:
comparables = ComparableValuation(yahoo)
peer_table = comparables.peer_table(PEERS)
display(peer_table)

if not peer_table.empty:
    implied_values = comparables.implied_values(snapshot, peer_table)
    display(implied_values)
else:
    print("No peer data returned.")